In [7]:
import PIL
from PIL import Image 
import numpy as np

In [8]:
loaded_image = Image.open(r"c:\Users\National\Pictures\reading.png", mode = 'r')
loaded_image.show()


In [ ]:
import cv2

# Load your ID template image
image_path = r"c:\Users\National\Pictures\reading.png"
img = cv2.imread(image_path)

print("--> Click anywhere on the image to get pixel coordinates.")
print("--> Press 'q' or 'ESC' to close the window when you are finished.")

def click_event(event, x, y, flags, params):
    # Check if the left mouse button was clicked
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"Clicked Coordinates -> X: {x}, Y: {y}")
        
        # Visually confirm by drawing a small red dot where you clicked
        cv2.circle(img, (x, y), 3, (0, 0, 255), -1)
        cv2.imshow("Template Anchor Locator", img)

# Create a named window and bind the mouse click function to it
cv2.namedWindow("Template Anchor Locator")
cv2.setMouseCallback("Template Anchor Locator", click_event)

# Keep the window open until 'q' or ESC is pressed
while True:
    cv2.imshow("Template Anchor Locator", img)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        break

cv2.destroyAllWindows()

--> Click anywhere on the image to get pixel coordinates.
--> Press 'q' or 'ESC' to close the window when you are finished.
Clicked Coordinates -> X: 464, Y: 93
Clicked Coordinates -> X: 464, Y: 111
Clicked Coordinates -> X: 505, Y: 112
Clicked Coordinates -> X: 505, Y: 90
Clicked Coordinates -> X: 362, Y: 124
Clicked Coordinates -> X: 361, Y: 142
Clicked Coordinates -> X: 504, Y: 142
Clicked Coordinates -> X: 504, Y: 119
Clicked Coordinates -> X: 506, Y: 159
Clicked Coordinates -> X: 472, Y: 160
Clicked Coordinates -> X: 472, Y: 181
Clicked Coordinates -> X: 507, Y: 182
Clicked Coordinates -> X: 508, Y: 195
Clicked Coordinates -> X: 336, Y: 190
Clicked Coordinates -> X: 336, Y: 215
Clicked Coordinates -> X: 509, Y: 214
Clicked Coordinates -> X: 225, Y: 258
Clicked Coordinates -> X: 513, Y: 257
Clicked Coordinates -> X: 513, Y: 286
Clicked Coordinates -> X: 229, Y: 286


KeyboardInterrupt: 

: 

In [10]:
import os
import random
from PIL import Image, ImageDraw, ImageFont

class EgyptianIDGenerator:
    def __init__(self, template_path, font_path, id_font_path=None):
        """
        template_path: Path to the clean front template (with no text).
        font_path: Path to an Arabic TTF font (e.g., Simplified Arabic Bold).
        id_font_path: Path to the specific font style for the 14-digit number.
        """
        self.template_path = template_path
        # Load fonts with specific sizes matching the document scale
        self.font_text = ImageFont.truetype(font_path, size=24)
        self.font_id = ImageFont.truetype(id_font_path or font_path, size=28)
        
        # Mapping Western digits to Eastern Arabic numerals used on the card
        self.to_eastern_digits = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
        
        # Sample pools for realistic data generation
        self.first_names = ["أحمد", "محمد", "محمود", "منير", "علي", "عمر", "مصطفى"]
        self.rest_names = ["شوكت أحمد", "كمال الدين حسن", "السيد عبد الله", "ابراهيم محمد"]
        self.addresses = ["مركز بنها - القليوبية", "١٢ شارع التحرير - الدقي - الجيزة", "مدينة نصر - القاهرة"]
        self.governorate_codes = ["٠١", "٠٢", "٢١", "١٤"] # Cairo, Alex, Giza, Qalyubia...

    def _generate_fake_id_number(self):
        """Generates a logically valid 14-digit Egyptian ID string in Eastern Arabic numerals"""
        century = random.choice(["٢", "٣"]) # 1900s vs 2000s
        year = f"{random.randint(0, 99):02d}".translate(self.to_eastern_digits)
        month = f"{random.randint(1, 12):02d}".translate(self.to_eastern_digits)
        day = f"{random.randint(1, 28):02d}".translate(self.to_eastern_digits)
        gov = random.choice(self.governorate_codes)
        sequence = f"{random.randint(1, 9999):04d}".translate(self.to_eastern_digits)
        chk = f"{random.randint(1, 9):01d}".translate(self.to_eastern_digits)
        
        return f"{century}{year}{month}{day}{gov}{sequence}{chk}"

    def generate_sample(self, output_path):
        # Open a fresh copy of the blank background template
        img = Image.open(self.template_path).convert("RGB")
        draw = ImageDraw.Draw(img)
        
        # 1. Generate fake data fields
        first_name = random.choice(self.first_names)
        full_name = random.choice(self.rest_names)
        addr = random.choice(self.addresses)
        national_id = self._generate_fake_id_number()
        
        # 2. Define anchor points (X, Y) matching your bounding boxes
        # Note: Since Arabic is Right-to-Left (RTL), we anchor text from the right if using 
        # advanced rendering, or calculate a right-aligned offset manually.
        # For simplicity here, these are standard top-left start positions:
        anchors = {
            "first_name": (450, 140),
            "full_name": (320, 180),
            "address": (300, 280),
            "national_id": (220, 420)
        }
        
        # 3. Draw text layers onto background image
        # Using direction="rtl" handles correct Arabic text shaping if pil has raqm support
        draw.text(anchors["first_name"], first_name, fill=(0, 0, 0), font=self.font_text, direction="rtl")
        draw.text(anchors["full_name"], full_name, fill=(0, 0, 0), font=self.font_text, direction="rtl")
        draw.text(anchors["address"], addr, fill=(0, 0, 0), font=self.font_text, direction="rtl")
        
        # Draw the 14-digit National ID track at the bottom
        draw.text(anchors["national_id"], national_id, fill=(0, 0, 0), font=self.font_id)
        
        # 4. Save the generated synthetic image
        img.save(output_path)
        
        # Return exact ground-truth coordinates along with labels for your model's training logs
        return {
            "image": output_path,
            "ground_truth": {
                "first_name": first_name,
                "full_name": full_name,
                "address": addr,
                "national_id": national_id
            }
        }

# Example Usage:
# generator = EgyptianIDGenerator("blank_front.png", "fonts/Simplified-Arabic-Bold.ttf")
# metadata = generator.generate_sample("dataset/train/sample_1.png")
# print(metadata)

In [1]:
def convert_to_yolo(img_width, img_height, bbox_pixels, class_id):
    """
    bbox_pixels: tuple from draw.textbbox -> (left, top, right, bottom)
    """
    left, top, right, bottom = bbox_pixels
    
    # Calculate raw width and height of the text zone
    box_width = right - left
    box_height = bottom - top
    
    # Calculate the center point of the box
    x_center = left + (box_width / 2.0)
    y_center = top + (box_height / 2.0)
    
    # Normalize coordinates by dividing by image dimensions
    x_center_norm = x_center / img_width
    y_center_norm = y_center / img_height
    width_norm = box_width / img_width
    height_norm = box_height / img_height
    
    return f"{class_id} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f}"

In [2]:
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

# 1. Load your clean image template
# (Make sure "egyptian_national_id.jpg" is in the same folder)
image_path = r"c:\Users\National\Pictures\reading.png"
img = Image.open(image_path)
draw = ImageDraw.Draw(img)

# 2. Load a standard font file 
# (You can copy 'arial.ttf' or 'times.ttf' from your computer's font folder into this project)
font_path = r"c:\Users\National\Downloads\Traditional Arabic Bold\Traditional Arabic Bold.ttf" 
font = ImageFont.truetype(font_path, size=22)

# 3. The Secret Sauce: Reshaping the Arabic Text
raw_text = "مركز بنها - القليوبية"
reshaped_text = arabic_reshaper.reshape(raw_text)    # Fixes the letter connections
final_text = get_display(reshaped_text)               # Fixes the Right-to-Left direction

# 4. Use your exact clicked coordinates!
# We will use the top-left of your Address Line 2 field: X=336, Y=190
text_position = (336, 190)

# 5. Draw it and save!
draw.text(text_position, final_text, fill=(0, 0, 0), font=font)
img.save(r"c:\Users\National\Pictures\test_output.png")

print("Done! Open 'test_output.png' to see your text perfectly placed.")

Done! Open 'test_output.png' to see your text perfectly placed.


In [26]:
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

# 1. Paths Configuration
image_path = r"c:\Users\National\Pictures\reading.png"
font_path = r"c:\Users\National\Downloads\Traditional Arabic Bold\Traditional Arabic Bold.ttf"
output_path = r"c:\Users\National\Pictures\test_output2.png"

# 2. Load the template using OpenCV for background cleaning
img_cv = cv2.imread(image_path)

# Extract a base clean slice from the empty center space to use as our "eraser" texture
clean_bg_slice = img_cv[190:215, 200:330] 

# Erase Box 1: First Name ((464, 90) to (505, 112) -> Width: 41, Height: 22)
img_cv[90:112, 464:505] = cv2.resize(clean_bg_slice, (41, 22))

# Erase Box 2: Rest of Legal Name ((361, 119) to (504, 142) -> Width: 143, Height: 23)
img_cv[119:142, 361:504] = cv2.resize(clean_bg_slice, (143, 23))

# Erase Box 3: Profession/Subfield ((472, 159) to (507, 182) -> Width: 35, Height: 23)
img_cv[159:182, 472:507] = cv2.resize(clean_bg_slice, (35, 23))

# Erase Box 4: Address Line 2 ((336, 190) to (509, 215) -> Width: 173, Height: 25)
img_cv[190:215, 336:509] = cv2.resize(clean_bg_slice, (173, 25))

# Erase Box 5: 14-Digit National ID ((225, 257) to (513, 286) -> Width: 288, Height: 29)
img_cv[257:286, 225:513] = cv2.resize(clean_bg_slice, (288, 29))


# 3. Convert back to PIL for high-quality text rendering
img = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
draw = ImageDraw.Draw(img)

# Load fonts with scaling relative to document fields
font_text = ImageFont.truetype(font_path, size=27)
font_id = ImageFont.truetype(font_path, size=26)

# Helper function to correct Arabic text rendering loops
def prepare_arabic_text(text):
    reshaped = arabic_reshaper.reshape(text)
    return get_display(reshaped)

# 4. Mock Data Assignments
txt_first_name  = prepare_arabic_text("يحيى")
txt_full_name   = prepare_arabic_text("محمد عادل نصار")
txt_address_l1  = prepare_arabic_text("المعادي")
txt_address_l2  = prepare_arabic_text("١٢ شارع التحرير - الدقي - الجيزة")
txt_national_id = "٣٠٣٠٦٢٢١٤٠١٢٣٤" 

# 5. Draw Text with Strict Uniform Right-Margin Alignment
# All text fields will terminate flush against this rightmost boundary line
COMMON_RIGHT_ALIGN_X = 505

# Render First Name (Y: 90)
w1 = draw.textlength(txt_first_name, font=font_text)
draw.text((COMMON_RIGHT_ALIGN_X - w1, 90), txt_first_name, fill=(0, 0, 0), font=font_text)

# Render Full Name (Y: 119)
w2 = draw.textlength(txt_full_name, font=font_text)
draw.text((COMMON_RIGHT_ALIGN_X - w2, 119), txt_full_name, fill=(0, 0, 0), font=font_text)

# Render Profession / Line 1 (Y: 159)
w3 = draw.textlength(txt_address_l1, font=font_text)
draw.text((COMMON_RIGHT_ALIGN_X - w3, 159), txt_address_l1, fill=(0, 0, 0), font=font_text)

# Render Address Details / Line 2 (Y: 190)
w4 = draw.textlength(txt_address_l2, font=font_text)
draw.text((COMMON_RIGHT_ALIGN_X - w4, 190), txt_address_l2, fill=(0, 0, 0), font=font_text)

# Render National ID (Stays left-anchored to match structural numeric sequence layout)
draw.text((225, 257), txt_national_id, fill=(0, 0, 0), font=font_id)


# 6. Save final output
img.save(output_path)
print(f"Success! Generated image saved directly to: {output_path}")



Success! Generated image saved directly to: c:\Users\National\Pictures\test_output2.png


In [36]:
import os
import cv2
import random
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
from faker import Faker

# Initialize Arabic data generator
fake = Faker('ar_EG') 

# --- CONFIGURATION PATHS ---
image_path  = r"c:\Users\National\Pictures\reading.png" # Update to your confirmed path
font_path   = r"c:\Users\National\Downloads\Traditional Arabic Bold\Traditional Arabic Bold.ttf"
output_dir  = r"c:\Users\National\Pictures\id_dataset"

# Create output directories for training if they don't exist
os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "labels"), exist_ok=True)

# Load base image to get static dimensions
img_test = cv2.imread(image_path)
if img_test is None:
    raise FileNotFoundError("Verify your input image path!")
img_h, img_w, _ = img_test.shape

# --- ARABIC TEXT ENGINE ---
font_text = ImageFont.truetype(font_path, size=27)
font_id   = ImageFont.truetype(font_path, size=26)

def prepare_arabic_text(text):
    return get_display(arabic_reshaper.reshape(text))

def to_eastern_arabic_numerals(number_str):
    western_to_eastern = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
    return number_str.translate(western_to_eastern)

# --- COORDINATES ENGINE ---
COMMON_RIGHT_ALIGN_X = 505

def write_and_get_bbox(draw, text, y_pos, font):
    """Draws right-aligned Arabic text and returns its standard YOLO normalized box parameters"""
    text_w = draw.textlength(text, font=font)
    x_pos = COMMON_RIGHT_ALIGN_X - text_w
    
    # Draw the text physically on the canvas
    draw.text((x_pos, y_pos), text, fill=(0, 0, 0), font=font)
    
    # Get strict pixel boundaries: (left, top, right, bottom)
    pixel_box = draw.textbbox((x_pos, y_pos), text, font=font)
    
    # Convert to YOLO standard format: [center_x, center_y, width, height] normalized by image size
    box_w = pixel_box[2] - pixel_box[0]
    box_h = pixel_box[3] - pixel_box[1]
    cx = pixel_box[0] + (box_w / 2.0)
    cy = pixel_box[1] + (box_h / 2.0)
    
    return cx / img_w, cy / img_h, box_w / img_w, box_h / img_h

# --- MAIN GENERATOR FACTORY LOOP ---
TOTAL_IMAGES_TO_GENERATE = 1  # Set this to 1000+ when ready to build your final dataset!

print(f"Generating {TOTAL_IMAGES_TO_GENERATE} synthetic training samples...")

for idx in range(TOTAL_IMAGES_TO_GENERATE):
    # Reload fresh OpenCV image every loop iteration to start with a blank template
    img_cv = cv2.imread(image_path)
    
    # 1. Clear out original document text fields using our verified patch layout
    clean_bg_slice = img_cv[190:215, 200:330] 
    img_cv[90:112, 464:505]   = cv2.resize(clean_bg_slice, (41, 22))    # Box 1
    img_cv[119:142, 361:504]  = cv2.resize(clean_bg_slice, (143, 23))   # Box 2
    img_cv[159:182, 472:507]  = cv2.resize(clean_bg_slice, (35, 23))    # Box 3
    img_cv[190:215, 336:509]  = cv2.resize(clean_bg_slice, (173, 25))   # Box 4
    img_cv[257:286, 225:513]  = cv2.resize(clean_bg_slice, (288, 29))   # Box 5

    # 2. Convert cleanly to PIL Canvas
    img = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    
    # 3. Generate randomized fake personal details
    first_name_raw = fake.first_name_male() if random.random() > 0.5 else fake.first_name_female()
    last_name_raw  = f"{fake.first_name_male()} {fake.first_name_male()}"
    address_raw    = f"شارع {fake.street_name()} - {fake.city()}"
    address2_raw   = f"{fake.city()} - {fake.state()}"
    
    # Create valid synthetic 14-digit Egyptian ID parameters
    century = random.choice(["2", "3"])
    birth_date = fake.date_of_birth(minimum_age=16, maximum_age=80).strftime("%y%m%d")
    gov_code = f"{random.randint(1, 28):02d}"
    seq_num = f"{random.randint(1, 9999):04d}"
    chk_digit = f"{random.randint(1, 9):01d}"
    raw_id_string = f"{century}{birth_date}{gov_code}{seq_num}{chk_digit}"

    # Shape text parameters
    txt_first_name = prepare_arabic_text(first_name_raw)
    txt_full_name  = prepare_arabic_text(last_name_raw)
    txt_address    = prepare_arabic_text(address_raw)
    txt_address2   = prepare_arabic_text(address2_raw)
    txt_nat_id     = to_eastern_arabic_numerals(raw_id_string)

    # 4. Draw Text and gather corresponding dataset YOLO annotations
    yolo_labels = []
    
    # Class 0: First Name (Y: 90)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_first_name, 90, font_text)
    yolo_labels.append(f"0 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 1: Full Name (Y: 119)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_full_name, 119, font_text)
    yolo_labels.append(f"1 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 2: Profession (Y: 159)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_address, 159, font_text)
    yolo_labels.append(f"2 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 3: Full Address Line 2 (Y: 190)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_address2, 190, font_text)
    yolo_labels.append(f"3 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 4: National ID (Static left-aligned numeric cluster)
    draw.text((225, 257), txt_nat_id, fill=(0, 0, 0), font=font_id)
    # Convert static ID coordinates manually into normalized YOLO formatting
    id_pixel_w, id_pixel_h = 288, 29
    id_cx = (225 + (id_pixel_w / 2.0)) / img_w
    id_cy = (257 + (id_pixel_h / 2.0)) / img_h
    yolo_labels.append(f"4 {id_cx:.6f} {id_cy:.6f} {(id_pixel_w/img_w):.6f} {(id_pixel_h/img_h):.6f}")

    # 5. Save the generated image asset and tracking label file pairs
    file_base_name = f"egyptian_id_sample_{idx:05d}"
    
    img.save(os.path.join(output_dir, "images", f"{file_base_name}.png"))
    with open(os.path.join(output_dir, "labels", f"{file_base_name}.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_labels))

print(f"Data Generation Completed! Files are organized in: {output_dir}")

Generating 1 synthetic training samples...
Data Generation Completed! Files are organized in: c:\Users\National\Pictures\id_dataset


In [40]:
# --- PURE ARABIC DATA GENERATION POOLS ---
arabic_first_names = ["أحمد", "محمد", "محمود", "منير", "مصطفى", "يوسف", "كريم", "عمرو", "علي", "سامح"]
arabic_last_names = [
    "شريف حسن", "منير شوكت", "كمال الدين أحمد", "عبد الرحمن علي", 
    "أحمد فاروق", "اشرف عبدالرحمان خالد", "سعيد مصطفى", "خالد حسين", "محمود عبد الله", "أحمد السقا",
    "عمرو عبدالنافع", "محمد عبد العزيز", "أحمد عبد الرحمن", "مصطفى عبد الله", "يوسف عبد الغني"
]
arabic_streets = ["شارع التحرير", "شارع جامعة الدول", "شارع عباس العقاد", "شارع الرازى", "شارع الأهرام"]
arabic_cities = ["الدقي - الجيزة", "مركز بنها - القليوبية", "مدينة نصر - القاهرة", "المعادي - القاهرة", "طنطا - الغربية"]

# --- MAIN GENERATOR FACTORY LOOP ---
TOTAL_IMAGES_TO_GENERATE = 50  # Set your target size

print(f"Generating {TOTAL_IMAGES_TO_GENERATE} synthetic training samples...")

for idx in range(TOTAL_IMAGES_TO_GENERATE):
    img_cv = cv2.imread(image_path)
    
    # 1. Clear out original document text fields using our verified patch layout
    clean_bg_slice = img_cv[190:215, 200:330] 
    img_cv[90:112, 464:505]   = cv2.resize(clean_bg_slice, (41, 22))    # Box 1
    img_cv[119:142, 361:504]  = cv2.resize(clean_bg_slice, (143, 23))   # Box 2
    img_cv[159:182, 472:507]  = cv2.resize(clean_bg_slice, (35, 23))    # Box 3
    img_cv[190:215, 336:509]  = cv2.resize(clean_bg_slice, (173, 25))   # Box 4
    img_cv[257:286, 225:513]  = cv2.resize(clean_bg_slice, (288, 29))   # Box 5

    # 2. Convert cleanly to PIL Canvas
    img = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    
    # 3. Pull randomized personal details from our explicit Arabic pools
    first_name_raw = random.choice(arabic_first_names)
    last_name_raw  = random.choice(arabic_last_names)
    address_raw    = random.choice(arabic_streets)
    address2_raw   = random.choice(arabic_cities)
    
    # Create valid synthetic 14-digit Egyptian ID parameters (Faker date is fine here)
    century = random.choice(["2", "3"])
    birth_date = fake.date_of_birth(minimum_age=16, maximum_age=80).strftime("%y%m%d")
    gov_code = f"{random.randint(1, 28):02d}"
    seq_num = f"{random.randint(1, 9999):04d}"
    chk_digit = f"{random.randint(1, 9):01d}"
    raw_id_string = f"{century} {birth_date} {gov_code} {seq_num} {chk_digit}"

    # Shape text parameters
    txt_first_name = prepare_arabic_text(first_name_raw)
    txt_full_name  = prepare_arabic_text(last_name_raw)
    txt_address    = prepare_arabic_text(address_raw)
    txt_address2   = prepare_arabic_text(address2_raw)
    txt_nat_id     = to_eastern_arabic_numerals(raw_id_string)

    # 4. Draw Text and gather corresponding dataset YOLO annotations
    yolo_labels = []
    
    # Class 0: First Name (Y: 90)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_first_name, 90, font_text)
    yolo_labels.append(f"0 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 1: Full Name (Y: 119)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_full_name, 119, font_text)
    yolo_labels.append(f"1 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 2: Address Line 1 / Profession slot (Y: 159)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_address, 159, font_text)
    yolo_labels.append(f"2 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 3: Full Address Line 2 (Y: 190)
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_address2, 190, font_text)
    yolo_labels.append(f"3 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")
    
    # Class 4: National ID (Static left-aligned numeric cluster)
    #draw.text((225, 257), txt_nat_id, fill=(0, 0, 0), font=font_id)
    #id_pixel_w, id_pixel_h = 288, 29
    #id_cx = (225 + (id_pixel_w / 2.0)) / img_w
    #id_cy = (257 + (id_pixel_h / 2.0)) / img_h
    cx, cy, cw, ch = write_and_get_bbox(draw, txt_nat_id, 257, font_text)
    yolo_labels.append(f"4 {cx:.6f} {cy:.6f} {cw:.6f} {ch:.6f}")

    # 5. Save output file pairs
    file_base_name = f"egyptian_id_sample_{idx:05d}"
    img.save(os.path.join(output_dir, "images", f"{file_base_name}.png"))
    with open(os.path.join(output_dir, "labels", f"{file_base_name}.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_labels))

Generating 50 synthetic training samples...
